PROJECT SETUP

Imports

In [0]:
from pyspark.sql.functions import (
    col, when, concat, concat_ws, lit, lpad, sum as spark_sum
)

Create schemas

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold")

DataFrame[]

BRONZE LAYER INGESTION

Read Orders CSV from Volume

In [0]:
orders_file_path = "/Volumes/workspace/bronze/landing/orders.csv"

orders_raw_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(orders_file_path)
)

orders_raw_df.show()
orders_raw_df.printSchema()

+--------+-----------+----------+--------+
|order_id|customer_id|product_id|quantity|
+--------+-----------+----------+--------+
|    1001|        101|      P001|       2|
|    1002|        102|      P002|       1|
|    1003|        101|      P003|       4|
|    1004|        103|      P001|       1|
|    1005|        104|      P002|       3|
|    1006|       NULL|      P001|       2|
|    1007|        101|      P002|      -5|
|    1008|        102|      NULL|       3|
+--------+-----------+----------+--------+

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)



Save Bronze Orders

In [0]:
orders_raw_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.bronze.orders_raw")

Read Products JSON from Volume

In [0]:
products_file_path = "/Volumes/workspace/bronze/landing/products.json"

products_raw_df = (
    spark.read
    .option("multiline", "true")
    .json(products_file_path)
)

products_raw_df.show()
products_raw_df.printSchema()

+----------------+--------------------+---+--------------------+------+----------+--------------------+
|        category|         description| id|               image| price|    rating|               title|
+----------------+--------------------+---+--------------------+------+----------+--------------------+
|  men's clothing|Your perfect pack...|  1|https://fakestore...|109.95|{120, 3.9}|Fjallraven - Fold...|
|  men's clothing|Slim-fitting styl...|  2|https://fakestore...|  22.3|{259, 4.1}|Mens Casual Premi...|
|  men's clothing|great outerwear j...|  3|https://fakestore...| 55.99|{500, 4.7}|  Mens Cotton Jacket|
|  men's clothing|The color could b...|  4|https://fakestore...| 15.99|{430, 2.1}|Mens Casual Slim Fit|
|        jewelery|From our Legends ...|  5|https://fakestore...| 695.0|{400, 4.6}|John Hardy Women'...|
|        jewelery|Satisfaction Guar...|  6|https://fakestore...| 168.0| {70, 3.9}|Solid Gold Petite...|
|        jewelery|Classic Created W...|  7|https://fakestore...|

Save Bronze Products

In [0]:
products_raw_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.bronze.products_raw")

Simulate Customer DB Source

In [0]:
customers_data = [
    (101, "John Smith", "john@gmail.com", "Delhi"),
    (102, "Mike Brown", "mike@gmail.com", "Mumbai"),
    (103, "David Wilson", "david@gmail.com", "Bangalore"),
    (104, "Alex Taylor", "alex@gmail.com", "Pune")
]

customers_df = spark.createDataFrame(
    customers_data,
    ["customer_id", "customer_name", "email", "city"]
)

customers_df.show()

+-----------+-------------+---------------+---------+
|customer_id|customer_name|          email|     city|
+-----------+-------------+---------------+---------+
|        101|   John Smith| john@gmail.com|    Delhi|
|        102|   Mike Brown| mike@gmail.com|   Mumbai|
|        103| David Wilson|david@gmail.com|Bangalore|
|        104|  Alex Taylor| alex@gmail.com|     Pune|
+-----------+-------------+---------------+---------+



Save Bronze Customers

In [0]:
customers_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.bronze.customers_raw")

SILVER LAYER VALIDATION

Load Bronze Tables

In [0]:
orders_bronze_df = spark.read.table("workspace.bronze.orders_raw")

products_bronze_df = spark.read.table("workspace.bronze.products_raw")

customers_bronze_df = spark.read.table("workspace.bronze.customers_raw")

Standardize Products for Silver

In [0]:
products_silver_df = (
    products_bronze_df
    .withColumnRenamed("id", "product_id")
    .withColumnRenamed("title", "product_name")
    .withColumn(
        "product_id",
        concat(
            lit("P"),
            lpad(col("product_id").cast("string"), 3, "0")
        )
    )
    .withColumn("product_rating", col("rating.rate"))
    .withColumn("rating_count", col("rating.count"))
    .select(
        "product_id",
        "product_name",
        "category",
        "price",
        "product_rating",
        "rating_count"
    )
)

products_silver_df.show()

+----------+--------------------+----------------+------+--------------+------------+
|product_id|        product_name|        category| price|product_rating|rating_count|
+----------+--------------------+----------------+------+--------------+------------+
|      P001|Fjallraven - Fold...|  men's clothing|109.95|           3.9|         120|
|      P002|Mens Casual Premi...|  men's clothing|  22.3|           4.1|         259|
|      P003|  Mens Cotton Jacket|  men's clothing| 55.99|           4.7|         500|
|      P004|Mens Casual Slim Fit|  men's clothing| 15.99|           2.1|         430|
|      P005|John Hardy Women'...|        jewelery| 695.0|           4.6|         400|
|      P006|Solid Gold Petite...|        jewelery| 168.0|           3.9|          70|
|      P007|White Gold Plated...|        jewelery|  9.99|           3.0|         400|
|      P008|Pierced Owl Rose ...|        jewelery| 10.99|           1.9|         100|
|      P009|WD 2TB Elements P...|     electronics|  64

ADVANCED REJECT FRAMEWORK

In [0]:
reject_framework_df = (
    orders_bronze_df
    .withColumn(
        "customer_error",
        when(col("customer_id").isNull(), "CUSTOMER_ID_NULL").otherwise(None)
    )
    .withColumn(
        "product_error",
        when(col("product_id").isNull(), "PRODUCT_ID_NULL").otherwise(None)
    )
    .withColumn(
        "quantity_error",
        when(col("quantity") <= 0, "INVALID_QUANTITY").otherwise(None)
    )
)

Customer Referential Check

In [0]:
customer_lookup_df = customers_bronze_df.select(
    col("customer_id").alias("master_customer_id")
)

reject_framework_df = (
    reject_framework_df
    .join(
        customer_lookup_df,
        reject_framework_df.customer_id == customer_lookup_df.master_customer_id,
        "left"
    )
    .withColumn(
        "customer_referential_error",
        when(
            (col("customer_id").isNotNull()) &
            (col("master_customer_id").isNull()),
            "CUSTOMER_NOT_FOUND"
        ).otherwise(None)
    )
)

Product Referential Check

In [0]:
product_lookup_df = products_silver_df.select(
    col("product_id").alias("master_product_id")
)

reject_framework_df = (
    reject_framework_df
    .join(
        product_lookup_df,
        reject_framework_df.product_id == product_lookup_df.master_product_id,
        "left"
    )
    .withColumn(
        "product_referential_error",
        when(
            (col("product_id").isNotNull()) &
            (col("master_product_id").isNull()),
            "PRODUCT_NOT_FOUND"
        ).otherwise(None)
    )
)

Create Rejection Reason

In [0]:
reject_framework_df = reject_framework_df.withColumn(
    "rejection_reason",
    concat_ws(
        ",",
        col("customer_error"),
        col("product_error"),
        col("quantity_error"),
        col("customer_referential_error"),
        col("product_referential_error")
    )
)

reject_framework_df.select(
    "order_id",
    "customer_id",
    "product_id",
    "quantity",
    "rejection_reason"
).show(truncate=False)

+--------+-----------+----------+--------+----------------+
|order_id|customer_id|product_id|quantity|rejection_reason|
+--------+-----------+----------+--------+----------------+
|1001    |101        |P001      |2       |                |
|1002    |102        |P002      |1       |                |
|1003    |101        |P003      |4       |                |
|1004    |103        |P001      |1       |                |
|1005    |104        |P002      |3       |                |
|1006    |NULL       |P001      |2       |CUSTOMER_ID_NULL|
|1007    |101        |P002      |-5      |INVALID_QUANTITY|
|1008    |102        |NULL      |3       |PRODUCT_ID_NULL |
+--------+-----------+----------+--------+----------------+



Split Valid and Reject Orders

In [0]:
valid_orders_df = (
    reject_framework_df
    .filter(col("rejection_reason") == "")
    .select(
        "order_id",
        "customer_id",
        "product_id",
        "quantity"
    )
)

reject_orders_df = (
    reject_framework_df
    .filter(col("rejection_reason") != "")
    .select(
        "order_id",
        "customer_id",
        "product_id",
        "quantity",
        "rejection_reason"
    )
)

valid_orders_df.show()
reject_orders_df.show(truncate=False)

+--------+-----------+----------+--------+
|order_id|customer_id|product_id|quantity|
+--------+-----------+----------+--------+
|    1001|        101|      P001|       2|
|    1002|        102|      P002|       1|
|    1003|        101|      P003|       4|
|    1004|        103|      P001|       1|
|    1005|        104|      P002|       3|
+--------+-----------+----------+--------+

+--------+-----------+----------+--------+----------------+
|order_id|customer_id|product_id|quantity|rejection_reason|
+--------+-----------+----------+--------+----------------+
|1006    |NULL       |P001      |2       |CUSTOMER_ID_NULL|
|1007    |101        |P002      |-5      |INVALID_QUANTITY|
|1008    |102        |NULL      |3       |PRODUCT_ID_NULL |
+--------+-----------+----------+--------+----------------+



Save Silver Valid and Reject Tables

In [0]:
valid_orders_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver.orders_valid")

reject_orders_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver.orders_reject")

Create Silver Order Details

In [0]:
silver_order_details_df = (
    valid_orders_df
    .join(customers_bronze_df, on="customer_id", how="inner")
    .join(products_silver_df, on="product_id", how="inner")
    .select(
        "order_id",
        "customer_id",
        "customer_name",
        "city",
        "product_id",
        "product_name",
        "category",
        "price",
        "quantity",
        "product_rating",
        "rating_count"
    )
)

silver_order_details_df.show()

+--------+-----------+-------------+---------+----------+--------------------+--------------+------+--------+--------------+------------+
|order_id|customer_id|customer_name|     city|product_id|        product_name|      category| price|quantity|product_rating|rating_count|
+--------+-----------+-------------+---------+----------+--------------------+--------------+------+--------+--------------+------------+
|    1001|        101|   John Smith|    Delhi|      P001|Fjallraven - Fold...|men's clothing|109.95|       2|           3.9|         120|
|    1002|        102|   Mike Brown|   Mumbai|      P002|Mens Casual Premi...|men's clothing|  22.3|       1|           4.1|         259|
|    1003|        101|   John Smith|    Delhi|      P003|  Mens Cotton Jacket|men's clothing| 55.99|       4|           4.7|         500|
|    1004|        103| David Wilson|Bangalore|      P001|Fjallraven - Fold...|men's clothing|109.95|       1|           3.9|         120|
|    1005|        104|  Alex Taylo

Save Silver Order Details

In [0]:
silver_order_details_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver.order_details")

GOLD LAYER AGGREGATION

Read Silver Table for Gold Layer

In [0]:
silver_order_details_df = spark.read.table(
    "workspace.silver.order_details"
)

Create Gold Order Fact

In [0]:
gold_orders_df = (
    silver_order_details_df
    .withColumn(
        "total_amount",
        col("quantity") * col("price")
    )
)

gold_orders_df.show()

+--------+-----------+-------------+---------+----------+--------------------+--------------+------+--------+--------------+------------+------------+
|order_id|customer_id|customer_name|     city|product_id|        product_name|      category| price|quantity|product_rating|rating_count|total_amount|
+--------+-----------+-------------+---------+----------+--------------------+--------------+------+--------+--------------+------------+------------+
|    1001|        101|   John Smith|    Delhi|      P001|Fjallraven - Fold...|men's clothing|109.95|       2|           3.9|         120|       219.9|
|    1002|        102|   Mike Brown|   Mumbai|      P002|Mens Casual Premi...|men's clothing|  22.3|       1|           4.1|         259|        22.3|
|    1003|        101|   John Smith|    Delhi|      P003|  Mens Cotton Jacket|men's clothing| 55.99|       4|           4.7|         500|      223.96|
|    1004|        103| David Wilson|Bangalore|      P001|Fjallraven - Fold...|men's clothing|1

Save Gold Order Fact

In [0]:
gold_orders_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.order_fact")

Create Gold City Sales Summary

In [0]:
city_sales_df = (
    gold_orders_df
    .groupBy("city")
    .agg(
        spark_sum("total_amount").alias("total_revenue")
    )
)

city_sales_df.show()

+---------+-------------+
|     city|total_revenue|
+---------+-------------+
|    Delhi|       443.86|
|   Mumbai|         22.3|
|Bangalore|       109.95|
|     Pune|         66.9|
+---------+-------------+



Save Gold City Sales Summary

In [0]:
city_sales_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.city_sales_summary")

Create Gold Category Sales Summary

In [0]:
category_sales_df = (
    gold_orders_df
    .groupBy("category")
    .agg(
        spark_sum("total_amount").alias("total_revenue")
    )
)

category_sales_df.show()

+--------------+-------------+
|      category|total_revenue|
+--------------+-------------+
|men's clothing|       643.01|
+--------------+-------------+



Save Gold Category Sales Summary

In [0]:
category_sales_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.category_sales_summary")

FINAL TABLE VERIFICATION

In [0]:
spark.sql("SHOW TABLES IN workspace.bronze").show()
spark.sql("SHOW TABLES IN workspace.silver").show()
spark.sql("SHOW TABLES IN workspace.gold").show()

+--------+-------------+-----------+
|database|    tableName|isTemporary|
+--------+-------------+-----------+
|  bronze|customers_raw|      false|
|  bronze|   orders_raw|      false|
|  bronze| products_raw|      false|
+--------+-------------+-----------+

+--------+-------------+-----------+
|database|    tableName|isTemporary|
+--------+-------------+-----------+
|  silver|order_details|      false|
|  silver|orders_reject|      false|
|  silver| orders_valid|      false|
+--------+-------------+-----------+

+--------+--------------------+-----------+
|database|           tableName|isTemporary|
+--------+--------------------+-----------+
|    gold|category_sales_su...|      false|
|    gold|  city_sales_summary|      false|
|    gold|          order_fact|      false|
+--------+--------------------+-----------+

